# t4llm — one T4, crash-only training

Run the cells in order. Nothing here writes the HF token to disk, and no cell
publishes a model until `export.license`, the repo names and every dataset
licence are confirmed.

Before a real run, fill in `configs/base.yaml`: `run.hub_repo`, `run.data_repo`,
`run.data_revision` and the base model. The current defaults are placeholders.


In [ ]:
# 1. Bootstrap: pinned deps, capability report, CPU contracts.
# Stop if the report does not say TRAINING-CAPABLE.
!bash scripts/bootstrap.sh


In [ ]:
# 2. Token: Colab Secret named HF_TOKEN, or a hidden prompt.
# Never print it, never write it to a file, never commit it.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN (input hidden): ")
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))


## 3. CPT data

Input JSONL: one document per line with `text`, `source`, `license`.
A missing/unknown licence stops the build. The output revision is immutable and
`index.json` is uploaded last.


In [ ]:
# Edit the input path first. This builds locally; add --push once the
# dataset repo exists and is confirmed writable.
!python -m t4llm data --config configs/cpt.yaml   --input /content/corpus.jsonl --out data


In [ ]:
# Pin the printed revision in configs/base.yaml (run.data_revision and
# data.revision), then verify the local index.
from t4llm.data import load_index
i = load_index("data")
print(i["data_revision"], i["total_tokens"], len(i["shards"]))


## 4. CPT

`supervise` owns a child process and a wall-clock budget. A `kill -9` loses at
most one checkpoint interval; `LATEST` moves only after a complete, verified
upload.


In [ ]:
!python -m t4llm supervise --config configs/cpt.yaml --time-budget 20m


In [ ]:
!python -m t4llm status --config configs/cpt.yaml


## 5. SFT

JSONL records use either `messages` (last turn assistant) or
`prompt` + `response`, plus `source` and `license`. Prompt tokens are masked;
packing is refused; NEFTune is active at `neftune_noise_alpha: 5.0`.


In [ ]:
# Point this at a licence-declaring SFT JSONL.
SFT_FILE = "/content/sft.jsonl"
!python -m t4llm sft --config configs/sft.yaml --train-file "$SFT_FILE"


In [ ]:
!python -m t4llm status --config configs/sft.yaml --train-file "$SFT_FILE"


## 6. DPO

JSONL records need `prompt`, `chosen`, `rejected` and `license`. Only the
standard sigmoid objective is shipped; the reference policy is the same model
with the adapter disabled.


In [ ]:
DPO_FILE = "/content/dpo.jsonl"
!python -m t4llm dpo --config configs/dpo.yaml --train-file "$DPO_FILE"


## 7. Evaluation

Always scores the frozen base and the adapter on identical chunks. A missing
general set reports `not_measured`, never “no regression”. Exit code 2 means the
general-set guard tripped.


In [ ]:
!python -m t4llm eval --config configs/cpt.yaml   --domain-set /content/heldout.jsonl


## 8. Export

Merged fp16 safetensors, `SHA256SUMS`, model card, and f16 GGUF. Set
`export.license` before publishing; the default card says do not redistribute.


In [ ]:
!python -m t4llm export --config configs/cpt.yaml
# Local-only check without a Hub push:
# !python -m t4llm export --config configs/cpt.yaml --no-push --no-gguf


## 9. Source backup

Commits the source (not checkpoints, not data) to the private|public backup
repo. The token stays in the environment.


In [ ]:
# Change the repo name only after confirming it.
!git add -A && git commit -m "notebook run snapshot" || true
!python scripts/push_hf_backup.py --repo Ggboykxz/t4llm-source
